# Customer Behavior & Revenue Insights Analysis

This notebook supports the end-to-end workflow by loading the retail dataset, applying the shared cleaning pipeline, and generating business-ready customer views.


In [ ]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'customer_behavior_revenue_insights.py').exists() and (BASE_DIR / 'python').exists():
    BASE_DIR = BASE_DIR / 'python'

sys.path.insert(0, str(BASE_DIR))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from customer_behavior_revenue_insights import (
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    build_summary_tables,
    clean_customer_behavior_data,
    load_raw_data,
)

sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
raw_df = load_raw_data(RAW_DATA_PATH)
raw_df.head()

In [ ]:
clean_df = clean_customer_behavior_data(raw_df)
clean_df.head()

In [ ]:
clean_df[['customer_id', 'age_group', 'purchase_frequency_days', 'customer_lifetime_segment']].head(10)

In [ ]:
summary_tables = build_summary_tables(clean_df)
summary_tables['kpis']

In [ ]:
summary_tables['customer_lifetime_segments']

In [ ]:
summary_tables['top_locations']

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

sns.barplot(
    data=summary_tables['revenue_by_category'],
    x='category',
    y='total_revenue',
    hue='category',
    legend=False,
    palette='Blues_d',
    ax=axes[0]
)
axes[0].set_title('Revenue by Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Total Revenue')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(
    data=summary_tables['revenue_by_subscription_status'],
    x='subscription_status',
    y='average_spend',
    hue='subscription_status',
    legend=False,
    palette='Set2',
    ax=axes[1]
)
axes[1].set_title('Average Spend by Subscription Status')
axes[1].set_xlabel('Subscription Status')
axes[1].set_ylabel('Average Spend')

sns.barplot(
    data=summary_tables['customer_lifetime_segments'],
    x='customer_lifetime_segment',
    y='customer_count',
    hue='customer_lifetime_segment',
    legend=False,
    palette='viridis',
    ax=axes[2]
)
axes[2].set_title('Customer Lifetime Segments')
axes[2].set_xlabel('Lifetime Segment')
axes[2].set_ylabel('Customers')
axes[2].tick_params(axis='x', rotation=20)

sns.barplot(
    data=summary_tables['top_locations'],
    x='total_revenue',
    y='location',
    hue='location',
    legend=False,
    palette='magma',
    ax=axes[3]
)
axes[3].set_title('Top 10 Locations by Revenue')
axes[3].set_xlabel('Total Revenue')
axes[3].set_ylabel('Location')

plt.tight_layout()
plt.show()

In [ ]:
output_path = save_processed_data(clean_df, PROCESSED_DATA_PATH)
output_path

## Optional Database Load

Use the Python script for database loading so credentials stay outside the notebook:

```bash
python3 python/customer_behavior_revenue_insights.py --load-to-db --db-type postgresql --db-user <username> --db-password <password>
```

The pipeline keeps the target table name as `customer`, which preserves the downstream SQL and Power BI logic.
